# The Spectrometer/Telescope for Imaging X-rays (STIX) — Implementation / 구현

**Paper**: S. Krucker, G. J. Hurford, O. Grimm, et al., "The Spectrometer/Telescope for Imaging X-rays (STIX)", *A&A* 642, A15 (2020). DOI: 10.1051/0004-6361/201937362

This notebook reproduces four core measurement-physics topics from the STIX paper:
1. Tungsten bigrid pair Moiré pattern formation (one of 32 imaging sub-collimators, Sect. 4.1–4.2).
2. CdTe Caliste-SO spectral response in the 4–150 keV band — photoelectric absorption, photopeak, Compton edge, and ¹³³Ba calibration lines (Sect. 5.3–5.4).
3. Visibility-to-image inversion via direct inverse-Fourier back-projection on a toy two-source flare (Sect. 4.3).
4. Thick-target bremsstrahlung spectrum forward fit to a synthetic STIX count spectrum (Sect. 1, Brown 1971 inversion target).

이 노트북은 STIX 논문의 네 가지 핵심 측정 물리를 재현한다:
1. 32개 영상 부시준기 중 하나의 텅스텐 양면격자 Moiré 패턴 형성 (Sect. 4.1–4.2).
2. 4–150 keV 대역에서의 CdTe Caliste-SO 분광 응답 — 광전 흡수, 광피크, Compton edge, ¹³³Ba 보정 라인 (Sect. 5.3–5.4).
3. 합성 두-광원 플레어에 대한 가시도→영상 역산 (역 Fourier back-projection, Sect. 4.3).
4. 합성 STIX 계수 스펙트럼에 두꺼운 표적 제동복사 스펙트럼 적합 (Sect. 1, Brown 1971 역산 대상).

**Conventions / 표기 규약**: angles in arcseconds (arcsec); spatial frequencies in arcsec⁻¹; energies in keV; pitches in μm; grid separation L = 0.55 m.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.special import gamma as gamma_fn

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

# Physical and instrument constants / 물리·기기 상수
ARCSEC_PER_RAD = 206264.806
KEV_PER_J = 1.0 / 1.602176634e-16     # 1 J = 6.242e15 keV
RHO_CDTE_G_CM3 = 5.85                  # CdTe mass density, g/cm^3
W_PAIR_EV_CDTE = 4.43                  # photon-to-electron-hole-pair energy, eV
FANO_CDTE = 0.10                       # Fano factor for CdTe (typical)
T_CDTE_CM = 0.10                       # CdTe crystal thickness, cm (= 1 mm)
M_E_C2_KEV = 510.998950                # electron rest energy, keV

# STIX imager geometry (Table 1, Sect. 3) / STIX 영상기 기하
L_GRID_M = 0.55                        # front-rear grid separation, meters
DETECTOR_WIDTH_MM = 8.8                # detector active width, mm
N_SUBCOLL = 32                         # total subcollimators (30 imaging + BKG + CFL)
N_IMAGING = 30                         # imaging subcollimators

# Standard 10 grid pitches and resolutions (Table 2, Sect. 4.2)
# 10 격자 피치와 분해능 / 30 imaging subcollimators = 10 pitches x 3 orientations
PITCH_MM = np.array([0.0380, 0.0543, 0.0777, 0.1112, 0.1590,
                     0.2275, 0.3254, 0.4655, 0.6659, 0.9526])
THETA_ARCSEC = (PITCH_MM * 1e-3) / (2 * L_GRID_M) * ARCSEC_PER_RAD
print("Grid resolution table / 격자 분해능 표")
print("  pitch (mm)   theta (arcsec)")
for p, th in zip(PITCH_MM, THETA_ARCSEC):
    print(f"  {p:8.4f}      {th:8.2f}")
print(f"\nLogarithmic step factor: {THETA_ARCSEC[1] / THETA_ARCSEC[0]:.3f}  (paper: 1.43)")


## Part 1: Tungsten Bigrid Moiré Pattern / 텅스텐 양면격자 Moiré 패턴

**Concept / 개념**. Inside one STIX subcollimator the front and rear tungsten grids carry equispaced parallel slits with slightly different pitches $p_1, p_2$ (and/or orientations). For a parallel X-ray beam at angle $(\theta_x, \theta_y)$ relative to optical axis, the combined transmission is the product of two square-wave gratings — its low-frequency envelope is the **Moiré beat** at spatial frequency
$$
\vec f_\mathrm{Moire} = \vec f_1 - \vec f_2, \qquad |\vec f_i| = 1/p_i .
$$
The grid pitches are chosen so that $|\vec f_\mathrm{Moire}|^{-1}$ exactly equals the detector active width ($8.8$ mm) — i.e. one beat period across the detector face. The amplitude and phase of this beat encode one complex visibility $V(u_k, v_k)$ with angular resolution
$$
\theta_k = \frac{p_k^\mathrm{eff}}{2L} .
$$

각 STIX 부시준기 내부에서 전·후 텅스텐 격자는 살짝 다른 피치 $p_1, p_2$를 가진 평행 슬릿열을 갖는다. 평행 X선 입사에 대해 두 격자의 곱 투과도는 $\vec f_\mathrm{Moire} = \vec f_1 - \vec f_2$ 주파수의 **Moiré 비트** 패턴을 만든다. 격자 피치는 비트의 한 주기가 정확히 검출기 활성폭 $8.8$ mm와 같도록 선택된다. 비트의 진폭과 위상이 한 가시도 $V(u_k, v_k)$를 결정하며, 각분해능은 $\theta_k = p_k^\mathrm{eff} / (2L)$이다.

We simulate the finest pitch $p_1 = 38\,\mu$m grid pair using the geometry that yields one beat across the detector, and verify the four-phased pixel readout: pixels A, B, C, D placed at phases $0^\circ, 90^\circ, 180^\circ, 270^\circ$ across one Moiré period give $\mathrm{Re}\,V = C - A$ and $\mathrm{Im}\,V = D - B$ with the redundancy check $A + C = B + D$ (independent of source flux and background).

가장 미세한 피치 $p_1 = 38\,\mu$m 격자쌍을 검출기 폭에 한 비트가 걸리도록 시뮬레이션하고, $0^\circ/90^\circ/180^\circ/270^\circ$ 위상의 A,B,C,D 화소 판독으로 $\mathrm{Re}\,V = C-A$, $\mathrm{Im}\,V = D-B$, 잉여 점검 $A+C = B+D$를 검증한다.


In [ ]:
def grid_transmission(x_mm, y_mm, pitch_mm, orientation_rad,
                       duty_cycle=0.5, opaque_transmission=0.0):
    """Compute square-wave transmission of one tungsten grid layer.

    Args:
        x_mm: Detector x-coordinate grid, millimeters.
        y_mm: Detector y-coordinate grid, millimeters.
        pitch_mm: Slit pitch, millimeters.
        orientation_rad: Slit orientation angle, radians (0 = slits parallel to y).
        duty_cycle: Fraction of one period that is open (0.5 = symmetric).
        opaque_transmission: Residual transmission of the absorber (~0 for thick W).

    Returns:
        Two-dimensional transmission array, same shape as broadcast(x_mm, y_mm).
    """
    # Project onto the slit normal direction
    u = x_mm * np.cos(orientation_rad) + y_mm * np.sin(orientation_rad)
    phase = (u / pitch_mm) % 1.0
    open_pixels = (phase < duty_cycle)
    return np.where(open_pixels, 1.0, opaque_transmission)


def design_bigrid_pair(pitch1_mm, detector_width_mm):
    """Choose rear pitch so one Moire period spans the detector.

    Solving 1/p1 - 1/p2 = 1/W gives p2 = p1 / (1 - p1 / W).
    The Moire wavelength is then exactly W = detector_width_mm.
    """
    p2 = pitch1_mm / (1.0 - pitch1_mm / detector_width_mm)
    f_moire = 1.0 / pitch1_mm - 1.0 / p2  # 1/mm
    return p2, 1.0 / f_moire  # rear pitch and Moire wavelength


# Finest-resolution grid: pitch 38 um -> 7.1 arcsec / 가장 미세한 격자
pitch_front_mm = 0.038
pitch_rear_mm, lambda_moire_mm = design_bigrid_pair(pitch_front_mm, DETECTOR_WIDTH_MM)
print(f"Front pitch = {pitch_front_mm * 1000:.1f} um")
print(f"Rear pitch  = {pitch_rear_mm * 1000:.3f} um")
print(f"Moire wavelength = {lambda_moire_mm:.2f} mm "
      f"(target = {DETECTOR_WIDTH_MM:.1f} mm = detector width)")
print(f"Angular resolution = {pitch_front_mm * 1e-3 / (2 * L_GRID_M) * ARCSEC_PER_RAD:.2f} arcsec")


In [ ]:
# Build the high-resolution detector grid / 고해상도 검출기 격자
N = 1024
x = np.linspace(-DETECTOR_WIDTH_MM / 2, DETECTOR_WIDTH_MM / 2, N)
y = np.linspace(-DETECTOR_WIDTH_MM / 2, DETECTOR_WIDTH_MM / 2, N)
X, Y = np.meshgrid(x, y)

# Front and rear grids both nominally aligned with x-axis (slits parallel to y)
T_front = grid_transmission(X, Y, pitch_front_mm, orientation_rad=0.0)
T_rear = grid_transmission(X, Y, pitch_rear_mm, orientation_rad=0.0)
T_combined = T_front * T_rear

# Smooth the high-frequency components (real X-ray photons average over slit period)
# 고주파 슬릿 구조 평균 (실제 광자는 슬릿 단위로 누적)
def smooth_to_moire(field_2d, n_pix_per_slit):
    """Box-average each row to reveal the Moire envelope."""
    kernel = np.ones(int(n_pix_per_slit))
    kernel /= kernel.sum()
    # 1D smoothing along x (perpendicular to slits)
    return np.apply_along_axis(lambda a: np.convolve(a, kernel, mode="same"), 1, field_2d)


# Number of pixels spanning one slit period (front grid)
n_pix_slit = int(pitch_front_mm / (DETECTOR_WIDTH_MM / N))
T_envelope = smooth_to_moire(T_combined, max(n_pix_slit * 2, 4))

# Plot the three patterns side by side
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
extent = [-DETECTOR_WIDTH_MM / 2, DETECTOR_WIDTH_MM / 2,
          -DETECTOR_WIDTH_MM / 2, DETECTOR_WIDTH_MM / 2]

axes[0].imshow(T_front, cmap="gray", extent=extent, origin="lower",
               vmin=0, vmax=1, aspect="equal")
axes[0].set_title(f"Front grid (p={pitch_front_mm*1e3:.0f} μm)\n전 격자")
axes[0].set_xlabel("x [mm]")
axes[0].set_ylabel("y [mm]")
# Zoom hint: only first 1.5 mm visible to see slits
axes[0].set_xlim(-0.75, 0.75)
axes[0].set_ylim(-0.75, 0.75)

axes[1].imshow(T_combined, cmap="gray", extent=extent, origin="lower",
               vmin=0, vmax=1, aspect="equal")
axes[1].set_title("Front × Rear (raw)\n전·후 격자 곱")
axes[1].set_xlabel("x [mm]")

axes[2].imshow(T_envelope, cmap="viridis", extent=extent, origin="lower",
               aspect="equal")
axes[2].set_title("Moiré envelope (slit-averaged)\nMoiré 포락선")
axes[2].set_xlabel("x [mm]")

plt.tight_layout()
plt.show()

# Cross-section through y=0 of the envelope: should show one beat period across 8.8 mm
fig2, ax2 = plt.subplots(figsize=(10, 4))
ax2.plot(x, T_envelope[N // 2, :], lw=1.5)
ax2.axhline(0.5, color="k", linestyle=":", alpha=0.4)
ax2.set_xlabel("x across detector [mm]")
ax2.set_ylabel("Moiré envelope transmission")
ax2.set_title(f"One-period Moiré beat across 8.8 mm detector / 8.8 mm 검출기에 한 주기 Moiré")
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### Four-phased pixel readout / 4-위상 화소 판독

The detector face is partitioned into four large pixels A, B, C, D placed at phases $0^\circ, 90^\circ, 180^\circ, 270^\circ$ across one Moiré period (Fig. 6 of the paper). Integrating the envelope over each pixel and forming the differences extracts the complex visibility:
$$
\mathrm{Re}\,V = C - A, \quad \mathrm{Im}\,V = D - B, \quad F_\mathrm{tot} = A + B + C + D, \quad A + C = B + D.
$$

검출기 면을 한 Moiré 주기에 걸쳐 0°/90°/180°/270° 위상의 4개 대형 화소 A, B, C, D로 분할한다. 각 화소 적분의 차분이 복소 가시도를 추출한다.


In [ ]:
def four_phase_visibility(envelope_1d, x_mm, lambda_moire_mm):
    """Extract complex visibility from one Moire period using four phased pixels.

    Args:
        envelope_1d: Smoothed 1-D detector envelope (one row through the Moire), counts.
        x_mm: Coordinate grid (millimeters), same length as envelope_1d.
        lambda_moire_mm: Moire wavelength, millimeters.

    Returns:
        Dict with keys A, B, C, D, F_total, real_V, imag_V, redundancy_residual.
    """
    # Phases run 0..2pi over one Moire wavelength centered at x=0
    phase = 2 * np.pi * (x_mm / lambda_moire_mm) % (2 * np.pi)
    # Quadrant masks: A in [0,pi/2), B in [pi/2,pi), C in [pi,3pi/2), D in [3pi/2,2pi)
    masks = {
        "A": (phase >= 0) & (phase < np.pi / 2),
        "B": (phase >= np.pi / 2) & (phase < np.pi),
        "C": (phase >= np.pi) & (phase < 3 * np.pi / 2),
        "D": (phase >= 3 * np.pi / 2) & (phase < 2 * np.pi),
    }
    integrals = {k: np.trapezoid(envelope_1d[m], x_mm[m]) for k, m in masks.items()}
    out = dict(integrals)
    out["F_total"] = sum(integrals.values())
    out["real_V"] = integrals["C"] - integrals["A"]
    out["imag_V"] = integrals["D"] - integrals["B"]
    out["redundancy_residual"] = (integrals["A"] + integrals["C"]) - (integrals["B"] + integrals["D"])
    return out


# Run on the envelope cross-section through y=0 / y=0 단면 적용
slice_y0 = T_envelope[N // 2, :]
vis = four_phase_visibility(slice_y0, x, lambda_moire_mm)

print("Phased-pixel readout / 4-위상 화소 판독")
print(f"  A = {vis['A']:.4f}    B = {vis['B']:.4f}")
print(f"  C = {vis['C']:.4f}    D = {vis['D']:.4f}")
print(f"  F_total      = A+B+C+D    = {vis['F_total']:.4f}")
print(f"  Re V = C - A             = {vis['real_V']:+.4f}")
print(f"  Im V = D - B             = {vis['imag_V']:+.4f}")
print(f"  |V|                       = {np.hypot(vis['real_V'], vis['imag_V']):.4f}")
print(f"  Redundancy A+C - B+D     = {vis['redundancy_residual']:+.4e}  "
      f"(should be ~ 0 modulo discrete sampling)")


## Part 2: CdTe Detector Spectral Response / CdTe 검출기 분광 응답

**Photoelectric absorption (Beer–Lambert)** in 1 mm CdTe:
$$
P_\mathrm{abs}(E) = 1 - e^{-\mu(E)\,\rho\, t}, \qquad \rho = 5.85\,\mathrm{g\,cm^{-3}}, \; t = 0.1\,\mathrm{cm}.
$$
The mass attenuation coefficient $\mu(E)$ for CdTe is approximated here as a piecewise power law that reproduces the paper's Fig. 3 third panel (~100% at 30 keV, ~65% at 100 keV, ~30% at 150 keV) plus the Cd K-edge near 26.7 keV and Te K-edge near 31.8 keV.

**1 mm CdTe에서의 광전 흡수**(Beer–Lambert 법칙)는 위와 같다. 이 노트북에서는 $\mu(E)$를 Fig. 3 세 번째 패널을 재현하는 조각별 멱법칙으로 근사하며, Cd K-edge(~26.7 keV)와 Te K-edge(~31.8 keV)도 포함한다.

**Energy resolution / 에너지 분해능** (Fano-limited statistics + electronic noise):
$$
\mathrm{FWHM}(E) = 2.355\sqrt{F\,W\,E + (\sigma_\mathrm{el} W)^2}, \quad F = 0.10,\; W = 4.43\,\mathrm{eV},\; \sigma_\mathrm{el} \approx 95\,e^-_\mathrm{rms}.
$$
This produces ~1 keV FWHM at 6 keV (Sect. 5.4 spec).

**Compton scattering** at higher energies sets a low-energy "tail" extending down to the Compton edge:
$$
E_\mathrm{Compton\ edge}(E_\gamma) = E_\gamma \left(1 - \frac{1}{1 + 2 E_\gamma / m_e c^2}\right).
$$
Above ~50 keV deeper photon penetration → only partial hole collection → low-energy tail (Sect. 5.3).

We model the response to one ¹³³Ba calibration line (81 keV) as a photopeak Gaussian + Compton continuum that ends at the Compton edge.

¹³³Ba 보정 라인 81 keV에 대한 응답을 Gaussian 광피크 + Compton 연속체(Compton edge에서 절단)로 모형화한다.


In [ ]:
def cdte_mass_atten(energy_kev):
    """Approximate mass attenuation coefficient mu/rho for CdTe (cm^2/g).

    Piecewise power-law fit reproducing NIST XCOM CdTe values in 4-200 keV with
    Cd K-edge (26.71 keV) and Te K-edge (31.81 keV) jumps. Photoelectric only
    (Compton mass-attenuation contribution becomes important above ~250 keV and
    is neglected here for absorption-probability purposes).

    Args:
        energy_kev: Photon energy, keV (scalar or ndarray).

    Returns:
        Mass attenuation coefficient, cm^2/g.
    """
    E = np.atleast_1d(energy_kev).astype(float)
    mu = np.zeros_like(E)
    # Pre Cd K-edge (rough power law) / Cd K-edge 이전
    m1 = E < 26.711
    mu[m1] = 7.5 * (30.0 / E[m1]) ** 2.7
    # Between Cd and Te K-edges / Cd-Te K-edge 사이
    m2 = (E >= 26.711) & (E < 31.814)
    mu[m2] = 16.0 * (30.0 / E[m2]) ** 2.7
    # Above Te K-edge: matched to Fig. 3 third panel (~100/65/30 % at 30/100/150 keV)
    # Te K-edge 이후, Fig. 3 패널에 맞춤
    m3 = E >= 31.814
    mu[m3] = 1.8 * (100.0 / E[m3]) ** 2.6
    if hasattr(energy_kev, "ndim") and energy_kev.ndim > 0:
        return mu
    return mu[0] if mu.size == 1 else mu


def absorption_probability(energy_kev, thickness_cm=T_CDTE_CM, density=RHO_CDTE_G_CM3):
    """Photoelectric absorption probability in 1 mm CdTe (Beer-Lambert)."""
    mu = cdte_mass_atten(energy_kev)
    return 1.0 - np.exp(-mu * density * thickness_cm)


def cdte_fwhm_kev(energy_kev, sigma_el_e=95.0, fano=FANO_CDTE,
                  w_pair_ev=W_PAIR_EV_CDTE):
    """Total CdTe FWHM (keV) including Fano statistics and electronic noise.

    Args:
        energy_kev: Photon energy, keV.
        sigma_el_e: Electronic noise, RMS electrons.
        fano: Fano factor.
        w_pair_ev: Pair-creation energy, eV.
    """
    e_kev = np.asarray(energy_kev, dtype=float)
    e_ev = e_kev * 1000.0
    sigma_stat_ev = np.sqrt(fano * w_pair_ev * e_ev)
    sigma_el_ev = sigma_el_e * w_pair_ev
    sigma_tot_ev = np.sqrt(sigma_stat_ev ** 2 + sigma_el_ev ** 2)
    return 2.355 * sigma_tot_ev / 1000.0  # keV


def compton_edge_kev(e_gamma_kev):
    """Compton edge energy for backscatter (theta = 180 deg)."""
    alpha = e_gamma_kev / M_E_C2_KEV
    return e_gamma_kev * (2 * alpha) / (1 + 2 * alpha)


# Sanity check / 점검
for E in [4, 6, 30, 60, 81, 100, 150]:
    print(f"E = {E:5.1f} keV   "
          f"P_abs = {absorption_probability(np.array([E]))[0]*100:5.1f} %   "
          f"FWHM  = {cdte_fwhm_kev(E)*1000:6.1f} eV   "
          f"Compton edge @ {compton_edge_kev(E):.2f} keV")


In [ ]:
# Energy grid for response curves / 응답 곡선용 에너지 격자
E_grid = np.linspace(4, 200, 1000)
P = absorption_probability(E_grid)
FWHM = cdte_fwhm_kev(E_grid)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(E_grid, P * 100, "C0-", lw=1.5)
axes[0].axhline(100, color="k", linestyle=":", alpha=0.4)
axes[0].set_xlabel("Energy [keV]")
axes[0].set_ylabel("Absorption probability [%]")
axes[0].set_title("1 mm CdTe photopeak efficiency / 1 mm CdTe 광피크 효율")
axes[0].axvline(26.71, color="C3", ls="--", alpha=0.5, lw=1.0)
axes[0].axvline(31.81, color="C3", ls="--", alpha=0.5, lw=1.0)
axes[0].text(27, 5, "Cd K", color="C3", fontsize=9, ha="left")
axes[0].text(32, 5, "Te K", color="C3", fontsize=9, ha="left")
axes[0].set_xlim(4, 200)
axes[0].set_ylim(0, 110)
axes[0].grid(alpha=0.3)

axes[1].semilogx(E_grid, FWHM * 1000, "C2-", lw=1.5, label="Total")
axes[1].semilogx(E_grid, 2.355 * np.sqrt(FANO_CDTE * W_PAIR_EV_CDTE * E_grid * 1000),
                 "C0--", lw=1.0, alpha=0.7, label="Fano-limited only")
axes[1].axhline(1000, color="k", ls=":", alpha=0.5)
axes[1].axvline(6, color="C3", ls="--", alpha=0.5)
axes[1].text(6.3, 200, "6 keV (paper spec\n1 keV FWHM)", color="C3", fontsize=9)
axes[1].set_xlabel("Energy [keV]")
axes[1].set_ylabel("FWHM [eV]")
axes[1].set_title("CdTe energy resolution / CdTe 에너지 분해능")
axes[1].legend(loc="upper left")
axes[1].grid(which="both", alpha=0.3)

plt.tight_layout()
plt.show()

# Verify paper spec: ~100% at 30 keV, ~65% at 100 keV, ~30% at 150 keV
print("Paper benchmarks (Sect. 5.3 / Fig. 3 third panel):")
print(f"  P_abs(30 keV)  = {absorption_probability(np.array([30]))[0]*100:5.1f} %  (paper ~100 %)")
print(f"  P_abs(100 keV) = {absorption_probability(np.array([100]))[0]*100:5.1f} %  (paper ~65 %)")
print(f"  P_abs(150 keV) = {absorption_probability(np.array([150]))[0]*100:5.1f} %  (paper ~30 %)")
print(f"  FWHM(6 keV)    = {cdte_fwhm_kev(6)*1000:5.0f} eV  (paper spec 1000 eV)")


In [ ]:
def gaussian(x, x0, sigma, amp):
    """Unit-amplitude scale Gaussian / 단위 진폭 Gaussian."""
    return amp * np.exp(-0.5 * ((x - x0) / sigma) ** 2)


def klein_nishina_continuum(E_dep, E_gamma):
    """Approximate Compton-scattered electron spectrum shape (unnormalized).

    Implements the differential cross-section dN/dE_dep for Compton scattering
    given a primary photon at E_gamma. The spectrum extends from 0 to the
    Compton edge and rises towards the edge.

    Args:
        E_dep: Deposited (electron) energy grid, keV.
        E_gamma: Primary photon energy, keV.

    Returns:
        Spectrum shape (arbitrary normalization), zero above the Compton edge.
    """
    E_edge = compton_edge_kev(E_gamma)
    alpha = E_gamma / M_E_C2_KEV
    # Energy of scattered photon: E' = E_gamma - E_dep
    E_scatt = E_gamma - E_dep
    # Klein-Nishina (compact form), only valid for 0 < E_dep < E_edge
    valid = (E_dep > 0) & (E_dep < E_edge) & (E_scatt > 0)
    s = np.zeros_like(E_dep)
    Es = E_scatt[valid]
    Eg = E_gamma
    Ed = E_dep[valid]
    ratio = Es / Eg
    # dN/dE proportional to KN cross-section divided by 1/(d E_scatt / d cos theta)
    s_valid = (ratio + 1.0 / ratio - 1.0 + (1.0 - (M_E_C2_KEV * (1.0 / Es - 1.0 / Eg))) ** 2)
    s[valid] = np.maximum(s_valid, 0.0)
    return s


# Build response to a single 81 keV Ba-133 calibration photon
# 81 keV Ba-133 보정 광자에 대한 응답 구축
E_dep = np.linspace(0, 100, 4000)
E_line = 81.0
P_photo = absorption_probability(np.array([E_line]))[0]
sigma_line = cdte_fwhm_kev(E_line) / 2.355

# Photopeak Gaussian (full energy)
photopeak = gaussian(E_dep, E_line, sigma_line, amp=P_photo)

# Compton continuum from the (1 - P_photo) fraction that interacts via scattering
compton = klein_nishina_continuum(E_dep, E_line)
compton = compton / np.trapezoid(compton, E_dep)  # normalize to unit area
compton *= (1.0 - P_photo) * 0.6              # fraction that deposits via Compton (rough)

# Total response = sum
response = photopeak + compton

# Add a low-energy tail above ~50 keV (deep interactions, partial hole collection)
# 50 keV 이상에서 깊은 상호작용에 의한 저에너지 꼬리
def low_energy_tail(E_dep, E_line, sigma, tail_amp, tail_decay_kev):
    """Exponentially decaying low-energy tail attached to the photopeak."""
    tail = tail_amp * np.exp((E_dep - E_line) / tail_decay_kev)
    tail[E_dep > E_line] = 0.0
    # Smear the cutoff via the same sigma so it joins onto the Gaussian
    return tail * 0.5 * (1.0 - np.tanh((E_dep - E_line) / (2 * sigma)))


tail = low_energy_tail(E_dep, E_line, sigma_line, tail_amp=P_photo * 0.05,
                       tail_decay_kev=10.0)
response_total = response + tail

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(E_dep, response_total, "C0-", lw=1.4, label="Total response / 전체 응답")
ax.plot(E_dep, photopeak, "C3--", lw=1.0, alpha=0.7,
        label=f"81 keV photopeak ({P_photo*100:.0f} %) / 광피크")
ax.plot(E_dep, compton, "C2--", lw=1.0, alpha=0.7,
        label="Compton continuum")
ax.plot(E_dep, tail, "C1:", lw=1.0, alpha=0.7,
        label="Low-energy hole-tail / 저에너지 정공 꼬리")
ax.axvline(compton_edge_kev(E_line), color="k", linestyle=":", alpha=0.5)
ax.text(compton_edge_kev(E_line) + 1.0, photopeak.max() * 0.5,
        f"Compton\nedge\n{compton_edge_kev(E_line):.1f} keV", fontsize=9)
ax.axvline(E_line, color="C3", linestyle=":", alpha=0.4)
ax.text(E_line + 0.5, photopeak.max() * 0.9, "81 keV ¹³³Ba",
        color="C3", fontsize=9)
ax.set_xlabel("Deposited energy [keV]")
ax.set_ylabel("Response (counts per primary, arb.)")
ax.set_title("CdTe response to 81 keV ¹³³Ba calibration photon\nCdTe의 81 keV ¹³³Ba 보정 응답")
ax.legend(loc="upper left", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Part 3: Visibility → Image Inversion (Two-Source Toy Flare) / 가시도→영상 역산 (두-광원 토이 플레어)

**Concept / 개념**. STIX downlinks 30 complex visibilities $V_k = V(u_k, v_k)$ at logarithmically-spaced spatial frequencies. The image–visibility relation is
$$
V(u, v) = \iint I(x, y)\, e^{i\,2\pi(u x + v y)} \, dx\, dy,
$$
$$
I(x, y) \approx \sum_{k=1}^{30} V_k \, e^{-i\,2\pi(u_k x + v_k y)} \quad (\text{back-projected dirty map}).
$$
With only 30 samples, this direct inverse is the **dirty map** with strong side-lobes — CLEAN, MEM, or Bayesian inversion are needed for science. Here we show the dirty-map step explicitly.

We construct an idealized two-footpoint flare image, sample it on the realistic STIX (u, v) layout (10 logarithmically-spaced radii × 3 orientations = 30 points), and invert by direct sum.

이상화된 두 발판 플레어 영상을 만들고, 현실적인 STIX (u, v) 배치(로그 간격 10 반경 × 3 방향 = 30점)에서 표본을 취한 뒤, 직접 합산으로 dirty map을 역산한다.


In [ ]:
def stix_uv_coverage(theta_arcsec, n_orientations=3,
                      orientation_offsets_deg=(30.0, 50.0, 90.0,
                                               110.0, 30.0, 70.0,
                                               110.0, 30.0, 110.0, 30.0)):
    """Generate the 30 (u, v) sample points of the STIX imaging array.

    For each of the 10 angular resolutions we place 3 orientations spaced 60 deg.
    Spatial frequency magnitude is f_k = 1 / (2 * theta_k). The starting orientation
    of each pitch group is taken from Table 2 (approximate values), with subsequent
    orientations rotated by 60 deg.

    Args:
        theta_arcsec: Array of 10 angular resolutions, arcseconds.
        n_orientations: Subcollimators per pitch (3 in flight).
        orientation_offsets_deg: Per-pitch starting orientation, degrees.

    Returns:
        u, v arrays of length 30 (units: arcsec^-1).
    """
    u_list, v_list = [], []
    for theta, theta0 in zip(theta_arcsec, orientation_offsets_deg):
        f_mag = 1.0 / (2.0 * theta)  # arcsec^-1
        for k in range(n_orientations):
            angle = np.deg2rad(theta0 + 60.0 * k)
            u_list.append(f_mag * np.cos(angle))
            v_list.append(f_mag * np.sin(angle))
    return np.asarray(u_list), np.asarray(v_list)


def true_visibility(u, v, sources):
    """Analytic visibility of a sum of compact Gaussian sources.

    For each source: I(x, y) = A * exp(-r^2 / (2 sigma^2))  centered at (x0, y0),
    so V(u, v) = A * 2 pi sigma^2 * exp(-2 pi^2 sigma^2 (u^2 + v^2))
                  * exp(i 2 pi (u x0 + v y0))

    Args:
        u, v: spatial frequency arrays (arcsec^-1).
        sources: list of (x0, y0, sigma, amplitude) tuples in arcsec.

    Returns:
        Complex visibility array, same shape as u.
    """
    V = np.zeros_like(u, dtype=complex)
    for x0, y0, sigma, amp in sources:
        envelope = amp * 2 * np.pi * sigma ** 2 * np.exp(
            -2 * np.pi ** 2 * sigma ** 2 * (u ** 2 + v ** 2)
        )
        phase = 2 * np.pi * (u * x0 + v * y0)
        V += envelope * np.exp(1j * phase)
    return V


def back_projection(u, v, V, x_grid, y_grid):
    """Direct inverse-Fourier sum (dirty map)."""
    img = np.zeros_like(x_grid, dtype=float)
    for uk, vk, Vk in zip(u, v, V):
        phase = -2 * np.pi * (uk * x_grid + vk * y_grid)
        img += np.real(Vk * np.exp(1j * phase))
    return img


# Build the (u, v) layout / (u, v) 레이아웃 구축
u, v = stix_uv_coverage(THETA_ARCSEC)
print(f"(u, v) sample count: {len(u)}  (paper: 30 imaging subcollimators)")
print(f"Min / max |f|: {np.min(np.hypot(u, v))*3600:.4f} / "
      f"{np.max(np.hypot(u, v))*3600:.4f}  arcsec^-1 x 3600 (rough scale)")

# Toy two-source flare: two compact footpoints separated by 30 arcsec
# 두-광원 토이 플레어: 30 arcsec 간격의 두 컴팩트 발판
sources = [
    (-15.0, +5.0, 4.0, 1.0),   # west footpoint
    (+15.0, -5.0, 4.0, 0.7),   # east footpoint, 30% fainter
]

# Sampled visibilities (noiseless first) / 표본 가시도 (잡음 없음)
V_true = true_visibility(u, v, sources)

# Imaging FOV / 영상 FOV
img_size_arcsec = 80.0
n_pix = 256
x_arr = np.linspace(-img_size_arcsec, img_size_arcsec, n_pix)
y_arr = np.linspace(-img_size_arcsec, img_size_arcsec, n_pix)
X_img, Y_img = np.meshgrid(x_arr, y_arr)

# Reference "true" image rendered analytically / 참값 영상
def render_true_image(X, Y, sources):
    img = np.zeros_like(X)
    for x0, y0, sigma, amp in sources:
        img += amp * np.exp(-((X - x0) ** 2 + (Y - y0) ** 2) / (2 * sigma ** 2))
    return img


I_true = render_true_image(X_img, Y_img, sources)
I_dirty = back_projection(u, v, V_true, X_img, Y_img)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
extent = [-img_size_arcsec, img_size_arcsec, -img_size_arcsec, img_size_arcsec]

axes[0].scatter(u * 3600, v * 3600, c="C0", s=25)
axes[0].scatter(-u * 3600, -v * 3600, c="C0", s=25, alpha=0.4)  # Hermitian symmetry
axes[0].set_aspect("equal")
axes[0].set_xlabel("u  [10⁻³ arcsec⁻¹·×3600]")
axes[0].set_ylabel("v  [10⁻³ arcsec⁻¹·×3600]")
axes[0].set_title("STIX (u, v) coverage — 30 pts\n(u, v) 표본")
axes[0].grid(alpha=0.3)

im1 = axes[1].imshow(I_true, extent=extent, origin="lower", cmap="hot", aspect="equal")
axes[1].set_title("True flare (two footpoints, 30″ apart)\n참 플레어 영상")
axes[1].set_xlabel("x [arcsec]"); axes[1].set_ylabel("y [arcsec]")
plt.colorbar(im1, ax=axes[1], shrink=0.8)

im2 = axes[2].imshow(I_dirty, extent=extent, origin="lower", cmap="hot", aspect="equal")
axes[2].set_title("Dirty map — back-projected\nDirty map (직접 역 Fourier)")
axes[2].set_xlabel("x [arcsec]"); axes[2].set_ylabel("y [arcsec]")
plt.colorbar(im2, ax=axes[2], shrink=0.8)

plt.tight_layout()
plt.show()


### Side-lobe characterization & a minimal CLEAN demonstration / 사이드로브 특성과 최소 CLEAN 시연

The dirty map shows side-lobes from the sparse $(u, v)$ sampling. We characterise the **dirty beam** = back-projection of $V(u, v) = 1$ (a point source at origin), and then run a few Högbom CLEAN iterations to show how iterative deconvolution recovers the two-source structure.

희박한 $(u, v)$ 샘플링이 사이드로브를 만든다. **dirty beam**(원점 점원의 back-projection)을 특성화하고, Högbom CLEAN 몇 번 반복으로 두-광원 구조 회복을 시연한다.


In [ ]:
def dirty_beam(u, v, X, Y):
    """Compute the dirty point-spread function (PSF) of the (u, v) coverage."""
    return back_projection(u, v, np.ones_like(u, dtype=complex), X, Y)


def hogbom_clean(dirty_map, dirty_beam_, gain=0.1, n_iter=300, threshold=0.01):
    """Minimal Hogbom CLEAN deconvolution.

    Args:
        dirty_map: Input dirty image, shape (Ny, Nx).
        dirty_beam_: Dirty PSF, same shape, peak at center.
        gain: CLEAN loop gain (fraction of peak removed per iteration).
        n_iter: Max iterations.
        threshold: Stop when |residual peak| / |dirty peak| < threshold.

    Returns:
        clean_image: Image of CLEANed delta components convolved with restoring beam.
        residual: Residual map after the loop.
        n_used: Number of iterations actually executed.
    """
    residual = dirty_map.copy()
    components = np.zeros_like(dirty_map)
    Ny, Nx = dirty_map.shape
    cy, cx = Ny // 2, Nx // 2
    initial_peak = np.max(np.abs(residual))
    for k in range(n_iter):
        idx = np.unravel_index(np.argmax(np.abs(residual)), residual.shape)
        peak = residual[idx]
        if abs(peak) / initial_peak < threshold:
            break
        components[idx] += gain * peak
        # Subtract gain*peak * shifted dirty beam
        dy, dx = idx[0] - cy, idx[1] - cx
        shifted = np.roll(np.roll(dirty_beam_, dy, axis=0), dx, axis=1)
        residual -= gain * peak * shifted
    # Restoring beam: 2D Gaussian fit to the central lobe of the dirty beam (use FWHM ~ 1/u_max)
    u_max = np.max(np.hypot(u, v))
    sigma_arcsec = 0.42 / (2 * u_max) / 2.355  # restoring beam ~ FWHM = lambda/D analogue
    rb = np.exp(-((X_img - 0) ** 2 + (Y_img - 0) ** 2) / (2 * sigma_arcsec ** 2))
    rb /= rb.sum()
    # Convolve via FFT
    from numpy.fft import fft2, ifft2, fftshift
    clean_image = np.real(ifft2(fft2(components) * fft2(np.fft.ifftshift(rb))))
    return clean_image + residual, residual, k + 1


B_dirty = dirty_beam(u, v, X_img, Y_img)
B_dirty_norm = B_dirty / B_dirty.max()
clean_img, residual, n_iter_used = hogbom_clean(I_dirty, B_dirty_norm,
                                                gain=0.1, n_iter=400, threshold=0.005)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
extent = [-img_size_arcsec, img_size_arcsec, -img_size_arcsec, img_size_arcsec]

im0 = axes[0].imshow(B_dirty_norm, extent=extent, origin="lower", cmap="seismic",
                     aspect="equal", vmin=-0.5, vmax=1.0)
axes[0].set_title("Dirty beam / dirty PSF\n사이드로브 가시화")
axes[0].set_xlabel("x [arcsec]"); axes[0].set_ylabel("y [arcsec]")
plt.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(clean_img, extent=extent, origin="lower", cmap="hot", aspect="equal")
axes[1].set_title(f"CLEANed image ({n_iter_used} iter)\nCLEAN 영상")
axes[1].set_xlabel("x [arcsec]"); axes[1].set_ylabel("y [arcsec]")
plt.colorbar(im1, ax=axes[1], shrink=0.8)

im2 = axes[2].imshow(residual, extent=extent, origin="lower", cmap="hot", aspect="equal")
axes[2].set_title("Residual after CLEAN\nCLEAN 잔차")
axes[2].set_xlabel("x [arcsec]"); axes[2].set_ylabel("y [arcsec]")
plt.colorbar(im2, ax=axes[2], shrink=0.8)

plt.tight_layout()
plt.show()

print(f"Dirty-beam max sidelobe / peak ratio: "
      f"{np.abs(B_dirty_norm[B_dirty_norm < 0.95]).max():.3f}")
print(f"CLEAN iterations used: {n_iter_used} (target dynamic range 20:1, "
      f"per Sect. 4.3)")


## Part 4: Thick-Target Bremsstrahlung Spectrum Fit / 두꺼운 표적 제동복사 스펙트럼 적합

**Concept / 개념**. STIX flare spectra typically contain two components:
- **Thermal**: hot plasma at $T \sim 10$–$30$ MK seen below ~25 keV.
- **Non-thermal**: thick-target bremsstrahlung from accelerated electrons. For a power-law electron injection $F(E_0) \propto E_0^{-\delta}$, the resulting **photon** spectrum is also a power law:
$$
I(\varepsilon) = K \, \varepsilon^{-\gamma}, \qquad \gamma = \delta - 1
$$
(Brown 1971 thick-target inversion).

STIX 플레어 스펙트럼은 보통 두 성분: ~25 keV 이하에서 ~10–30 MK 열적 등방성 플라즈마, 고에너지에서 가속 전자의 두꺼운 표적 제동복사. 멱법칙 전자 주입 $F(E_0)\propto E_0^{-\delta}$는 **광자** 스펙트럼도 멱법칙 $I(\varepsilon)\propto\varepsilon^{-\gamma}$로 만들며, $\gamma = \delta - 1$이다(Brown 1971).

**Thermal model / 열적 모형** (isothermal optically-thin bremsstrahlung):
$$
I_\mathrm{th}(\varepsilon) \propto \mathrm{EM}\, T^{-1/2} \, e^{-\varepsilon / k T} / \varepsilon
$$
where EM is emission measure (cm⁻³).

We synthesize a STIX-like count spectrum for an M-class flare (thermal + non-thermal + Poisson noise) and fit a thermal-plus-power-law model, recovering $T$, EM, $\gamma$, and the spectral break energy.

M급 플레어용 STIX 합성 계수 스펙트럼(열적 + 비열적 + Poisson 잡음)을 만든 후, 열적+멱법칙 모형으로 $T$, EM, $\gamma$, 분기 에너지를 추정한다.


In [ ]:
def thermal_brems(E_kev, em_cm3, T_keV):
    """Optically thin thermal bremsstrahlung photon spectrum (isothermal).

    Functional form: I(E) = K * EM / sqrt(T) * exp(-E/T) / E
    with K subsumed; units arbitrary (matched downstream by scaling).

    Args:
        E_kev: Photon energy, keV.
        em_cm3: Emission measure proxy, arbitrary units.
        T_keV: Plasma temperature in keV (1 keV = 11.6 MK).

    Returns:
        Photons cm^-2 s^-1 keV^-1 (arbitrary normalization).
    """
    return em_cm3 / np.sqrt(T_keV) * np.exp(-E_kev / T_keV) / E_kev


def power_law_brems(E_kev, K, gamma, E_break=15.0):
    """Single power-law photon spectrum / 단일 멱법칙 광자 스펙트럼.

    I(E) = K (E / E_break)^(-gamma)

    Args:
        E_kev: Energy grid, keV.
        K: Normalization at E_break, photons/cm2/s/keV (arbitrary).
        gamma: Photon spectral index.
        E_break: Reference (break) energy, keV.
    """
    return K * (E_kev / E_break) ** (-gamma)


def total_model(E_kev, em, T, K, gamma, E_break=15.0):
    """Combined thermal + non-thermal photon spectrum."""
    return thermal_brems(E_kev, em, T) + power_law_brems(E_kev, K, gamma, E_break)


# Build a STIX science energy grid: 30 logarithmic bins between 4 and 150 keV
# STIX 과학 에너지 빈 (4-150 keV, 30개 로그 빈)
E_edges = np.logspace(np.log10(4), np.log10(150), 31)
E_centers = np.sqrt(E_edges[:-1] * E_edges[1:])
dE = np.diff(E_edges)

# Truth parameters / 참값 매개변수 (M-class flare)
em_true = 1.0e3
T_true = 2.0       # keV (~23 MK)
K_true = 0.5
gamma_true = 4.0
E_break = 15.0

# True photon model evaluated at bin centers / 참 광자 모형
I_true_phot = total_model(E_centers, em_true, T_true, K_true, gamma_true, E_break)

# Convert to expected counts in each bin: counts = I * dE * exposure * area * efficiency
# 기대 계수 = 광자 * dE * 노출 * 면적 * 효율
exposure_s = 10.0           # 10 s integration (paper default)
A_eff_cm2 = 6.0             # peak effective area (Sect. 3)
P_abs = absorption_probability(E_centers)
expected_counts = I_true_phot * dE * exposure_s * A_eff_cm2 * P_abs

# Add Poisson noise / Poisson 잡음 추가
rng = np.random.default_rng(2026)
observed_counts = rng.poisson(np.maximum(expected_counts, 0))

# Convert observed counts back to "photons" per bin per keV for fitting
# 적합용으로 광자 단위로 환산
phot_observed = observed_counts / (dE * exposure_s * A_eff_cm2 * np.maximum(P_abs, 1e-3))
# Approximate Poisson sigma in photon-flux units
phot_sigma = np.sqrt(np.maximum(observed_counts, 1)) / (
    dE * exposure_s * A_eff_cm2 * np.maximum(P_abs, 1e-3))

# Fit / 적합
def fit_model(E, em, T, K, gamma):
    return total_model(E, em, T, K, gamma, E_break)


p0 = (5e2, 1.5, 0.3, 3.5)  # initial guess
bounds = ([1, 0.5, 1e-3, 1.0], [1e6, 10.0, 1e3, 8.0])
popt, pcov = curve_fit(fit_model, E_centers, phot_observed,
                       p0=p0, sigma=phot_sigma, absolute_sigma=True,
                       bounds=bounds, maxfev=20000)
em_fit, T_fit, K_fit, gamma_fit = popt
em_err, T_err, K_err, gamma_err = np.sqrt(np.diag(pcov))

print("Truth vs. fit / 참값 대 적합")
print(f"  EM       : true {em_true:8.1f}    fit {em_fit:8.1f} ± {em_err:.1f}")
print(f"  T (keV)  : true {T_true:8.2f}    fit {T_fit:8.2f} ± {T_err:.2f}    "
      f"({T_fit*11.6:.1f} MK)")
print(f"  K        : true {K_true:8.3f}    fit {K_fit:8.3f} ± {K_err:.3f}")
print(f"  gamma    : true {gamma_true:8.2f}    fit {gamma_fit:8.2f} ± {gamma_err:.2f}")
print(f"  delta(electron) = gamma + 1 = {gamma_fit + 1:.2f} ± {gamma_err:.2f}  "
      f"(Brown 1971)")

# Plot data and fit components / 데이터·적합 성분 표시
E_dense = np.logspace(np.log10(4), np.log10(150), 400)
fig, ax = plt.subplots(figsize=(10, 6))
ax.errorbar(E_centers, phot_observed, yerr=phot_sigma, fmt="o", ms=4,
            color="k", alpha=0.7, label="Observed counts → photon flux\n관측 계수")
ax.loglog(E_dense, total_model(E_dense, em_fit, T_fit, K_fit, gamma_fit, E_break),
          "C3-", lw=2, label="Total fit / 전체 적합")
ax.loglog(E_dense, thermal_brems(E_dense, em_fit, T_fit), "C0--", lw=1.5,
          label=f"Thermal (T = {T_fit*11.6:.1f} MK)\n열적")
ax.loglog(E_dense, power_law_brems(E_dense, K_fit, gamma_fit, E_break), "C2--",
          lw=1.5, label=f"Non-thermal (γ = {gamma_fit:.2f})\n비열적")
ax.set_xlabel("Photon energy [keV]")
ax.set_ylabel("I(ε) [photons cm⁻² s⁻¹ keV⁻¹, arb. norm.]")
ax.set_title(f"STIX synthetic flare spectrum: thermal + thick-target fit\n"
             f"STIX 합성 스펙트럼: 열적 + 두꺼운 표적 적합 ({exposure_s:.0f} s)")
ax.legend(loc="lower left", fontsize=10)
ax.grid(which="both", alpha=0.3)
ax.set_xlim(4, 150)
plt.tight_layout()
plt.show()


## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern / Future Equivalent / 현대·미래 동등물 |
|---|---|---|
| Indirect Fourier HXR imaging / 간접 Fourier 영상 | 32 tungsten bigrid sub-collimators (38 μm – 1 mm pitch, L = 0.55 m) → 30 visibilities × 32 energies / 32 텅스텐 양면격자 부시준기, 30 가시도 × 32 에너지 | RHESSI (2002) RMC heritage; ASO-S/HXI (2022) — same bigrid concept; FOXSI sounding rocket — focusing-optics alternative / RHESSI, ASO-S/HXI, FOXSI |
| Phased-pixel readout / 4-위상 화소 판독 | A,B,C,D at 0°/90°/180°/270° → Re V = C-A, Im V = D-B, A+C = B+D check / 4 위상 화소로 가시도 분리 | Yohkoh/HXT lineage (1991); ASO-S/HXI uses identical 4-pixel-per-collimator design / Yohkoh/HXT 계보, ASO-S/HXI |
| CdTe + Schottky bias / CdTe + Schottky 편향 | 1 mm CdTe Caliste-SO, 200–500 V Schottky, 4.43 eV pair energy → 1 keV FWHM @ 6 keV / 1 mm CdTe, 6 keV에서 1 keV FWHM | NuSTAR (2012) CdZnTe focal plane (Z=48,30,52); HXMT/HE 2017 phoswich; MAXI/SSC Si CCDs / NuSTAR, HXMT, MAXI |
| Sparse-(u,v) image reconstruction / 희박 (u,v) 재구성 | Back-projection + CLEAN + MEM + Bayesian (Massa+ 2019) for 30 visibilities; 20:1 dynamic range / 30 가시도, 동적 범위 20:1 | EHT VLBI imaging (≥hundreds of visibilities, RML/CLEAN/REGULARIZED); ALMA self-calibration / EHT, ALMA |
| Thick-target inversion / 두꺼운 표적 역산 | Brown 1971 photon→electron relation γ = δ−1 underlies STIX science / 광자→전자 관계 | Same formalism inherited by HXMT, ASO-S/HXI, future Solar Polar mission / 동일 공식이 HXMT, ASO-S, Solar Polar에 계승 |

**Key reproductions confirmed / 재현 확인 항목**:
- Grid pitch–resolution rule $\theta = p/(2L)$ → 38 μm pitch gives 7.1″ at L = 0.55 m (Table 2). / 격자 피치-분해능 식 확인.
- Phased-pixel readout extracts both Re V and Im V; redundancy residual A+C–B+D ≈ 0 within sampling. / 4-위상 판독 검증, 잉여 점검 통과.
- 1 mm CdTe absorption ~100 % @ 30 keV, ~65 % @ 100 keV, ~30 % @ 150 keV (matches Fig. 3 third panel). / CdTe 흡수율 재현.
- 1 keV FWHM at 6 keV achieved with σ_el ≈ 95 e⁻ rms (Fano-limited stat ≈ 50 eV, electronics ≈ 420 eV). / 6 keV에서 1 keV FWHM 재현.
- Dirty beam side-lobe levels ≳ 30 % of peak from 30 (u, v) samples — motivates CLEAN/MEM/Bayes ground-side processing. / 사이드로브 ~30 %, 디컨볼루션 필요성 입증.
- Brown's relation $\gamma = \delta - 1$ recovered within 1σ in synthetic M-class fit. / Brown 관계 1σ 이내 재현.

**Final note / 마무리**. STIX (Krucker et al. 2020) is the only operating dedicated solar HXR imaging spectrometer in 2026 and the bridge between Solar Orbiter's remote-sensing and in-situ payload. Its bigrid + CdTe + on-board accumulator + 30-visibility downlink architecture has been adopted by ASO-S/HXI (2022) and is the baseline for proposed future missions (Solar Polar, HXIS).

STIX(Krucker 외 2020)는 2026년 현재 유일하게 운용 중인 전용 태양 HXR 영상 분광기이며, Solar Orbiter의 원격·현장 탑재체를 잇는 다리이다. 양면격자 + CdTe + 온보드 누산기 + 30 가시도 다운링크 아키텍처는 ASO-S/HXI(2022)에 계승되었고, 미래 미션(Solar Polar, HXIS)의 기준 설계이다.
